# XGBoost Weekly — All-in-one (đọc 2 file gốc → left-join\)
Thay cho `evaluate_s2_v5_weekly_xgboost_FIXED`. Sửa 3 lỗi của bản cũ:
1. **Left-join** giữ toàn bộ salinity (bản cũ `.dropna()` trên cột phổ → mất 68% dữ liệu).
2. **Target đúng**: hồi quy dùng `salinity_max_mean`, phân loại dùng `salinity_max_max`.
3. **Đọc đúng file** `weekly_salinity_with_quality.csv` (có cột chất lượng).

Đầu vào: `S2_..._daily...csv` (phổ daily 9 strategy) + `weekly_salinity_with_quality.csv`.
Strategy chốt: **BUF30_MNDWI_strict** (có bảng so sánh 9 strategy để justify, không quét 270 tổ hợp → tránh p-hacking).

In [ ]:
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from scipy.stats import spearmanr

# ================= CẤU HÌNH (chốt với leader) =================
S2_DAILY_CSV = "../../Data/xgboost/raw/S2_v5_weekly30_2020_2023_ALLBUF_daily.csv"
WSQ_CSV      = "../../Data/xgboost/processed/weekly_salinity_with_quality.csv"
STRATEGY     = "BUF30_MNDWI_strict"   # buffer 30m + MNDWI_strict (đã justify trong bảng citation)
WATER_FRAC_MIN = 0.3   # QC phổ: blank khi tỉ lệ nước < ngưỡng (KHÔNG xoá dòng)
N_WATER_MIN    = 3     # QC phổ: blank khi số pixel nước < ngưỡng
SALINITY_CAP   = 40.0  # cap outlier phi vật lý
def rmse(a,b): return np.sqrt(mean_squared_error(a,b))

## 1. Load 2 file

In [2]:
s2  = pd.read_csv(S2_DAILY_CSV)
wsq = pd.read_csv(WSQ_CSV, parse_dates=["date"])
s2["station_id"]  = s2["station_id"].astype(str).str.strip().str.upper()
wsq["station_id"] = wsq["station_id"].astype(str).str.strip().str.upper()
s2["date"] = pd.to_datetime(s2["date"], errors="coerce")
s2["week_date"] = s2["date"].dt.to_period("W-SUN").dt.end_time.dt.normalize()
wsq["date"] = wsq["date"].dt.normalize()
print("S2 daily:", s2.shape, "| strategies:", s2["strategy_id"].nunique())
print("Salinity+quality:", wsq.shape)

S2 daily: (22717, 62) | strategies: 9
Salinity+quality: (1960, 51)


## 2. Gộp phổ daily → weekly (theo strategy)
Median của các chỉ số theo (trạm, tuần). Hàm dùng lại cho cả bảng so sánh lẫn dataset chính.

In [3]:
FEAT_DAILY = [c for c in s2.columns if c.endswith("_median") and not c.endswith("_count")]

def weekly_spectral(strategy):
    d = s2[s2["strategy_id"] == strategy]
    agg = {c: "median" for c in FEAT_DAILY}
    agg.update({"date": "count", "water_frac_clear": "mean", "n_water_px": "median"})
    w = d.groupby(["station_id", "week_date"]).agg(agg).reset_index()
    return w.rename(columns={**{c: c+"_week" for c in FEAT_DAILY},
                             "date": "s2_obs_count",
                             "water_frac_clear": "s2_wfrac", "n_water_px": "s2_nwater"})
print(len(FEAT_DAILY), "feature phổ")

23 feature phổ


## 3. Bảng so sánh 9 strategy (bằng chứng chọn BUF30_MNDWI_strict)
Ngưỡng cố định, chỉ so 9 strategy — KHÔNG quét 270 tổ hợp (tránh chọn ngưỡng theo kết quả).
Cột `best_|spearman|` = tương quan phổ↔độ mặn mạnh nhất trong nhóm chỉ số chính.

In [4]:
rows = []
for st in sorted(s2["strategy_id"].dropna().unique()):
    w = weekly_spectral(st)
    m = wsq[["station_id","date","salinity_max_mean"]].merge(
        w, left_on=["station_id","date"], right_on=["station_id","week_date"], how="inner"
    ).dropna(subset=["salinity_max_mean"])
    best_r, best_f = -1, None
    for f in ["MNDWI_median_week","NDWI_median_week","Red_SWIR1_median_week","NDCI_median_week","BGRratio_median_week"]:
        if f in m and m[f].notna().sum() > 20:
            r,_ = spearmanr(m[f], m["salinity_max_mean"], nan_policy="omit")
            if abs(r) > best_r: best_r, best_f = abs(r), f.replace("_median_week","")
    rows.append({"strategy": st, "n_rows_join": len(m), "best_index": best_f, "best_abs_spearman": round(best_r,3)})
comparison = pd.DataFrame(rows)
print("Lưu ý: |spearman| đều nhỏ -> phổ là biến phụ trợ, đúng tinh thần đề tài.\n")
comparison

Lưu ý: |spearman| đều nhỏ -> phổ là biến phụ trợ, đúng tinh thần đề tài.



,strategy,n_rows_join,best_index,best_abs_spearman
0,BUF20_MNDWI_basic,978,NDCI,0.077
1,BUF20_MNDWI_strict,977,NDCI,0.072
2,BUF20_NDWI_basic,981,NDCI,0.082
3,BUF30_MNDWI_basic,1028,NDCI,0.115
4,BUF30_MNDWI_strict,1028,NDCI,0.114
5,BUF30_NDWI_basic,1035,NDCI,0.106
6,BUF50_MNDWI_basic,1113,NDCI,0.107
7,BUF50_MNDWI_strict,1112,NDCI,0.103
8,BUF50_NDWI_basic,1114,NDCI,0.118


## 4. Dataset chính — LEFT JOIN + QC phổ
Gộp phổ của **BUF30_MNDWI_strict**, blank spectral khi kém tin cậy (water_frac/n_water thấp) — **giữ nguyên dòng, chỉ để NaN**. Sau đó left-join vào toàn bộ salinity.

In [ ]:
w = weekly_spectral(STRATEGY)
SPEC_COLS = [c+"_week" for c in FEAT_DAILY]
bad = (w["s2_wfrac"] < WATER_FRAC_MIN) | (w["s2_nwater"] < N_WATER_MIN)
w.loc[bad, SPEC_COLS] = np.nan
print(f"QC phổ: blank {int(bad.sum())}/{len(w)} tuần-trạm kém tin cậy (giữ dòng, để NaN)")

df = wsq.merge(w, left_on=["station_id","date"], right_on=["station_id","week_date"], how="left")
df["s2_available"] = df["MNDWI_median_week"].notna().astype(int)
print(f"Dataset: {len(df)} dòng (giữ hết salinity) | phổ dùng được: {int(df['s2_available'].sum())} | phổ=NaN: {int((df['s2_available']==0).sum())}")
df.to_csv("../../Data/xgboost/processed/weekly_ml_dataset_v2_leftjoin.csv", index=False)
print("-> đã lưu weekly_ml_dataset_v2_leftjoin.csv")

QC phổ: blank 282/1881 tuần-trạm kém tin cậy (giữ dòng, để NaN)
Dataset: 1960 dòng (giữ hết salinity) | phổ dùng được: 899 | phổ=NaN: 1061
-> đã lưu weekly_ml_dataset_v2_leftjoin.csv
